<a href="https://colab.research.google.com/github/kathirkarthi72/HappySpending_ML_CoLab/blob/Future_Amount_Prediction_Gen_Pickle/HappySpending_Future_Amount_Prediction_Gen_Pickle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
googlesheeturl = 'https://docs.google.com/spreadsheets/d/1nNOwqYMqOpdb-JpB5hTIorQhegyD68BuQgZFIcY7Xog/edit?gid=2077114165#gid=2077114165'

# Folder and file paths
gdir_folder_path = "/content/drive/MyDrive/happy_spending_ml"
happy_spending_model_pkl = "happy_spending_model.pkl"
input_file_csv = "input_data.csv"
output_file_csv = "output_data.csv"

# Step 1 (Fixed): Connect Google Colab to Google Sheets

In [ ]:
# Install necessary libraries
!pip install --upgrade gspread pandas gspread_dataframe --quiet

# Authenticate user
import gspread
import pandas as pd
from gspread_dataframe import get_as_dataframe
from google.colab import auth
auth.authenticate_user()

# Create credentials and authorize
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Replace with your Google Sheet URL
sheet_url = googlesheeturl
spreadsheet = gc.open_by_url(sheet_url)

# Access the first worksheet
worksheet = spreadsheet.sheet1

# Convert worksheet to DataFrame
df = get_as_dataframe(worksheet)
print(df)



               Timestamp            Spend By                   Buy From  \
0     4/14/2024 22:50:27  Kathiresan Murugan  Near by shop in Velachery   
1      4/15/2024 9:46:33  Kathiresan Murugan  Near by shop in Velachery   
2     4/15/2024 10:40:52  Kathiresan Murugan             Reliance Smart   
3     4/15/2024 11:05:38  Kathiresan Murugan  Near by shop in Velachery   
4     4/15/2024 11:06:03  Kathiresan Murugan  Near by shop in Velachery   
...                  ...                 ...                        ...   
1374   4/15/2025 7:06:58  Kathiresan Murugan                  Velachery   
1375  4/15/2025 13:48:50  Jeyasudha Angusamy                  Velachery   
1376  4/15/2025 13:49:26  Jeyasudha Angusamy                  Velachery   
1377  4/15/2025 13:49:58  Jeyasudha Angusamy                  Velachery   
1378  4/15/2025 21:11:07  Jeyasudha Angusamy                  Velachery   

                Spend Category           Notes   Buy On  Pay By  Amount  \
0                       

Find any empty rows

Step 1: Check for Empty Values in Specific Columns

In [ ]:
# Drop columns we don't need
df_clean = df.drop(columns=['Notes', 'Unique Id'])

# Check if any column has missing (NaN) values in any row
empty_rows = df_clean.isnull().any(axis=1)

# Filter rows with any missing values
empty_rows_df = df_clean[empty_rows]

# Print the empty rows (rows with NaN values in at least one column)
print("Empty Row df")
print(empty_rows_df)
print(df_clean.describe)
df = df_clean

Empty Row df
Empty DataFrame
Columns: [Timestamp, Spend By, Buy From, Spend Category, Buy On, Pay By, Amount]
Index: []
<bound method NDFrame.describe of                Timestamp            Spend By                   Buy From  \
0     4/14/2024 22:50:27  Kathiresan Murugan  Near by shop in Velachery   
1      4/15/2024 9:46:33  Kathiresan Murugan  Near by shop in Velachery   
2     4/15/2024 10:40:52  Kathiresan Murugan             Reliance Smart   
3     4/15/2024 11:05:38  Kathiresan Murugan  Near by shop in Velachery   
4     4/15/2024 11:06:03  Kathiresan Murugan  Near by shop in Velachery   
...                  ...                 ...                        ...   
1374   4/15/2025 7:06:58  Kathiresan Murugan                  Velachery   
1375  4/15/2025 13:48:50  Jeyasudha Angusamy                  Velachery   
1376  4/15/2025 13:49:26  Jeyasudha Angusamy                  Velachery   
1377  4/15/2025 13:49:58  Jeyasudha Angusamy                  Velachery   
1378  4/15/2025 21:11

# Convert Timestamp to datetime

In [ ]:
# Convert Timestamp to datetime
df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'])

# Extract time features
df_clean['Year'] = df_clean['Timestamp'].dt.year
df_clean['Month'] = df_clean['Timestamp'].dt.month
df_clean['Day'] = df_clean['Timestamp'].dt.day
df_clean['Hour'] = df_clean['Timestamp'].dt.hour
df_clean['Minute'] = df_clean['Timestamp'].dt.minute
df_clean['DayOfWeek'] = df_clean['Timestamp'].dt.dayofweek
df_clean['IsWeekend'] = df_clean['DayOfWeek'].isin([5, 6]).astype(int)
df_clean['IsMonthStart'] = df_clean['Timestamp'].dt.is_month_start.astype(int)
df_clean['IsMonthEnd'] = df_clean['Timestamp'].dt.is_month_end.astype(int)
df_clean['WeekOfYear'] = df_clean['Timestamp'].dt.isocalendar().week
df_clean['Quarter'] = df_clean['Timestamp'].dt.quarter

# Drop original timestamp (optional)
# df_clean = df_clean.drop(columns=['Timestamp'])

# Step 2: Clean & Prepare the Data

In [ ]:
df.head()

,Timestamp,Spend By,Buy From,Spend Category,Buy On,Pay By,Amount,Year,Month,Day,Hour,Minute,DayOfWeek,IsWeekend,IsMonthStart,IsMonthEnd,WeekOfYear,Quarter
0,2024-04-14 22:50:27,Kathiresan Murugan,Near by shop in Velachery,Fruits,Offline,Cash,130.0,2024,4,14,22,50,6,1,0,0,15,2
1,2024-04-15 09:46:33,Kathiresan Murugan,Near by shop in Velachery,Grocery,Offline,Cash,20.0,2024,4,15,9,46,0,0,0,0,16,2
2,2024-04-15 10:40:52,Kathiresan Murugan,Reliance Smart,Dairy Products,Offline,Sodexo,80.0,2024,4,15,10,40,0,0,0,0,16,2
3,2024-04-15 11:05:38,Kathiresan Murugan,Near by shop in Velachery,Non - Veg,Offline,Cash,200.0,2024,4,15,11,5,0,0,0,0,16,2
4,2024-04-15 11:06:03,Kathiresan Murugan,Near by shop in Velachery,Grocery,Offline,Cash,20.0,2024,4,15,11,6,0,0,0,0,16,2


# Step 3.1: Encode Categorical Columns

Let’s encode using LabelEncoder (simple and clean for structured ML):

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 2. Convert Timestamp to datetime
df_clean['Timestamp'] = pd.to_datetime(df['Timestamp'])

# 3. Extract datetime features
df_clean['DayOfWeek'] = df_clean['Timestamp'].dt.dayofweek
df_clean['Hour'] = df_clean['Timestamp'].dt.hour
df_clean['Day'] = df_clean['Timestamp'].dt.day
df_clean['Month'] = df_clean['Timestamp'].dt.month

# 4. Drop original Timestamp column
df_clean.drop(columns=['Timestamp'], inplace=True)

# 5. Drop specified categorical columns
df_clean.drop(columns=['Spend By', 'Buy From', 'Spend Category', 'Buy On', 'Pay By'], inplace=True)

# 6. Label Encode remaining categorical columns
# Assuming there might be some categorical columns left after the drop

# 7. View final preprocessed data
df_clean.head(5)

# Optional: Print the description of the data
# print(df.describe())


,Amount,Year,Month,Day,Hour,Minute,DayOfWeek,IsWeekend,IsMonthStart,IsMonthEnd,WeekOfYear,Quarter
0,130.0,2024,4,14,22,50,6,1,0,0,15,2
1,20.0,2024,4,15,9,46,0,0,0,0,16,2
2,80.0,2024,4,15,10,40,0,0,0,0,16,2
3,200.0,2024,4,15,11,5,0,0,0,0,16,2
4,20.0,2024,4,15,11,6,0,0,0,0,16,2


In [ ]:
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# # 2. Convert Timestamp to datetime
# df_clean['Timestamp'] = pd.to_datetime(df['Timestamp'])

# # 3. Extract datetime features
# df_clean['DayOfWeek'] = df_clean['Timestamp'].dt.dayofweek
# df_clean['Hour'] = df_clean['Timestamp'].dt.hour
# df_clean['Day'] = df_clean['Timestamp'].dt.day
# df_clean['Month'] = df_clean['Timestamp'].dt.month

# # 4. Drop original Timestamp column
# df_clean.drop(columns=['Timestamp'], inplace=True)

# # 5. Label Encode categorical columns
# label_encoders = {}
# categorical_columns = ['Spend By', 'Buy From', 'Spend Category', 'Buy On', 'Pay By']

# for col in categorical_columns:
#     le = LabelEncoder()
#     df_clean[col] = le.fit_transform(df_clean[col])
#     label_encoders[col] = le  # Save encoder for later decoding

# # 6. View final preprocessed data
# df_clean.head(5)
# # print(df.describe())

# ---

# 🔜 Ready for Step 3: Future Amount Prediction?
Step 3.1: Define Features and Target

In [ ]:
X = df_clean.drop(columns=['Amount'])  # Features
y = df_clean['Amount']                # Target

input = X
output = y

print("✅ Input: Cleaned X Columns:", X.columns.tolist())

print("✅ X Shape:", X.shape)
print("✅ y Preview:\n", y.head())

print("✅ y Shape:", y.shape)

✅ Input: Cleaned X Columns: ['Year', 'Month', 'Day', 'Hour', 'Minute', 'DayOfWeek', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'WeekOfYear', 'Quarter', 'Amount_Per_Day', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'Amount_Per_Day', 'Amount^2', 'Amount IsWeekend', 'Amount IsMonthStart', 'Amount IsMonthEnd', 'Amount Amount_Per_Day', 'IsWeekend^2', 'IsWeekend IsMonthStart', 'IsWeekend IsMonthEnd', 'IsWeekend Amount_Per_Day', 'IsMonthStart^2', 'IsMonthStart IsMonthEnd', 'IsMonthStart Amount_Per_Day', 'IsMonthEnd^2', 'IsMonthEnd Amount_Per_Day', 'Amount_Per_Day^2']
✅ X Shape: (1379, 31)
✅ y Preview:
      Amount    Amount
0  0.004711  0.004711
1  0.003828  0.003828
2  0.004310  0.004310
3  0.005273  0.005273
4  0.003828  0.003828
✅ y Shape: (1379, 2)



# Input - Feature Name | Description
1. Year | Year of transaction
2. Month | Month of transaction
3. Day | Day of transaction
4. Hour | Hour of transaction
5. Minute | Minute of transaction
6. DayOfWeek | Weekday index (0=Monday)
7. IsWeekend | Binary flag if it’s a weekend
8. IsMonthStart | Binary flag for start of the month
9. IsMonthEnd | Binary flag for end of the month
10. WeekOfYear | Week number in the year
11. Quarter | Quarter of the year (1 to 4)


## Output - Column Name | Description | Type
1. Amount | Expense amount (₹₹₹) | float64



Step 3.2: Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Step 3.3: Train a Basic Model (e.g., Random Forest Regressor)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Train a basic Random Forest model
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")



MAE: 1679.52
RMSE: 5287.37


# Refinement Steps:
1. Outlier Handling:

In [ ]:
# Step 1: Filter out negative or null Amount values
df_filtered = df_clean[df_clean['Amount'] > 0].copy()

# Step 2: Apply log1p safely
import numpy as np
df_filtered['LogAmount'] = np.log1p(df_filtered['Amount'])

X = df_filtered.drop(columns=['Amount', 'LogAmount'])
y = df_filtered['LogAmount']


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Reverse the log transform for predictions
y_pred = np.expm1(y_pred)

# Evaluate the model
mae = mean_absolute_error(np.expm1(y_test), y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")


MAE: 667.02
RMSE: 3366.47


# Feture Engineering

Feature | Type | Why It’s Useful

IsWeekend | Binary | Spending behavior differs on weekends.

IsMonthStart / IsMonthEnd | Binary | People often spend more during these times.

WeekOfMonth | Categorical | Helps spot spending patterns across weeks.

Quarter | Categorical | Can expose seasonal shifts in spending.

Add to your DataFrame:

1. Encoding Categorical Variables

In [ ]:
# One-hot encoding categorical columns
# df_clean = pd.get_dummies(df_clean, columns=['Spend By', 'Buy From', 'Spend Category', 'Buy On', 'Pay By'])



2. Scaling Numerical Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_clean['Amount'] = scaler.fit_transform(df_clean[['Amount']])


3. Feature Creation

In [ ]:
df_clean['Amount_Per_Day'] = df_clean['Amount'] / df_clean['Day']


4. Finalizing the Features

In [ ]:
# Print the list of columns in your cleaned data
print(df_clean.columns)



Index(['Amount', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'DayOfWeek',
       'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'WeekOfYear', 'Quarter',
       'Amount_Per_Day'],
      dtype='object')


In [ ]:
columns_to_drop = ['Unique Id', 'Timestamp']
columns_to_drop = [col for col in columns_to_drop if col in df_clean.columns]

df_clean = df_clean.drop(columns=columns_to_drop)


Define Features and Target:

In [ ]:
X = df_clean.drop(columns=['Amount'])  # All features except the target column
y = df_clean['Amount']  # Target column


Handle Categorical Data (if necessary):

In [ ]:
X = pd.get_dummies(X, drop_first=True)  # One-hot encoding for categorical features


Scaling: For numerical columns, you can scale the features using StandardScaler or MinMaxScaler:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Step 4: Model Training
1. Split the Data i

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


2. Train the Model: Let's use a regression model (e.g., Linear Regression or Random Forest Regressor) to predict the Amount.
Example using Linear Regression:

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)


LinearRegression()

3. Make Predictions: After training the model, you can make predictions on the test set:

In [ ]:
y_pred = model.predict(X_test)


4. Evaluate the Model: We'll evaluate the model using metrics like Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Calculate MAE
mae = mean_absolute_error(y_test, y_pred)

# Calculate MSE
mse = mean_squared_error(y_test, y_pred)

# Calculate RMSE manually
rmse = mse ** 0.5

# Print the metrics
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

MAE: 0.01
RMSE: 0.01


Great! Now let's move forward with model tuning to further improve your model's performance. The main idea behind model tuning is to adjust various parameters (called hyperparameters) to achieve the best performance. Here's how you can approach this:

1. Hyperparameter Tuning:
We'll use techniques like Grid Search or Random Search to tune the hyperparameters of the model. These methods help find the best parameters for a given model.

2. Cross-Validation:
Before tuning, you might want to incorporate cross-validation to prevent overfitting and evaluate the model's performance on different splits of the data.

Let's begin by using GridSearchCV (for exhaustive search over specified hyperparameters) with RandomForestRegressor as an example. We'll also use cross-validation to ensure the model is not overfitting.

Steps:

Define the hyperparameters for the model.
Use GridSearchCV to search over the parameters.
Evaluate the performance on the validation set.
Code Example for Hyperparameter Tuning (using RandomForestRegressor):

In [ ]:
# Import necessary libraries
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Define your parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': randint(10, 200),  # Randomly select number of trees between 10 and 200
    'max_depth': randint(1, 20),  # Randomly select depth between 1 and 20
    'min_samples_split': randint(2, 20),  # Randomly select the minimum samples required to split a node
}

# Initialize the RandomForestRegressor
rf_regressor = RandomForestRegressor()

# Set up the RandomizedSearchCV object
random_search = RandomizedSearchCV(
    rf_regressor,
    param_distributions=param_dist,
    n_iter=10,  # Number of random combinations to try
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,  # Use all available CPU cores
    verbose=1  # Print progress
)

# Fit the model to the training data (make sure X_train, y_train are already defined)
random_search.fit(X_train, y_train)

# Print the best parameters found
print("Best parameters found: ", random_search.best_params_)

# Get the best model from RandomizedSearchCV
best_model = random_search.best_estimator_

# Predict on the test data
y_pred = best_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Print the metrics
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")



Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found:  {'max_depth': 15, 'min_samples_split': 4, 'n_estimators': 140}
MAE: 0.00
RMSE: 0.01


# Next Step: Model Validation
1. Split the Data: First, split your data into training and testing datasets.

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming 'df_clean' is your cleaned dataframe and 'Amount' is the target variable
X = df_clean.drop(columns=['Amount'])  # Features
y = df_clean['Amount']  # Target variable

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



2. Scale the Features: Use StandardScaler to scale the features, which is important for models like RandomForest to avoid biases based on feature scale.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit and transform the training data
X_train_scaled = scaler.fit_transform(X_train)

# Only transform the testing data (to avoid data leakage)
X_test_scaled = scaler.transform(X_test)


3. Cross-validation: Now that your data is split and scaled, you can proceed with the cross-validation as mentioned earlier.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Using cross-validation to validate the model
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')

# Convert negative MSE to positive RMSE
cv_rmse_scores = (-cv_scores) ** 0.5

# Display the cross-validation results
print("Cross-validation RMSE scores: ", cv_rmse_scores)
print("Average RMSE: ", cv_rmse_scores.mean())


Cross-validation RMSE scores:  [0.01748982 0.01124092 0.03940978 0.01099216 0.01023564]
Average RMSE:  0.017873664398285437


4. Evaluate on the Test Set: Finally, evaluate your model on the hold-out test set to get more insights into its generalization performance.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_test_pred = model.predict(X_test_scaled)

# Calculate MAE
mae_test = mean_absolute_error(y_test, y_test_pred)

# Manually calculate RMSE
mse_test = mean_squared_error(y_test, y_test_pred)  # MSE is by default squared
rmse_test = mse_test ** 0.5  # Take square root to get RMSE

# Calculate R² (explained variance)
r2_test = r2_score(y_test, y_test_pred)

# Display the evaluation metrics
print(f"Test MAE: {mae_test:.2f}")
print(f"Test RMSE: {rmse_test:.2f}")
print(f"Test R²: {r2_test:.2f}")


Test MAE: 0.01
Test RMSE: 0.01
Test R²: 0.78


# Let’s start with step 1: improving feature engineering:
Step 1: Address Missing Data and Add Any Features

In [ ]:
# Check for missing values
missing_data = df_clean.isnull().sum()
print(f"Missing data: \n{missing_data[missing_data > 0]}")

# If any columns have missing data, we can handle them (e.g., imputation or dropping rows/columns)
# Here, we can use mean imputation for simplicity (but consider other methods based on the data)
df_clean = df_clean.fillna(df_clean.mean())

# Add polynomial features if needed, for example, adding quadratic features of numerical columns
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
df_poly = poly.fit_transform(df_clean.select_dtypes(include=['float64', 'int64']))

# Combine the polynomial features with the original dataframe (if needed)
poly_df = pd.DataFrame(df_poly, columns=poly.get_feature_names_out(df_clean.select_dtypes(include=['float64', 'int64']).columns))
df_clean = pd.concat([df_clean, poly_df], axis=1)

# Now df_clean contains both original and polynomial features


Missing data: 
Series([], dtype: int64)


Step 2: Model Tuning using RandomizedSearchCV

Here’s how we’ll do the hyperparameter tuning:

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# Initialize the model
model = RandomForestRegressor(random_state=42)

# Define hyperparameters for tuning
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=50, cv=5, verbose=2, random_state=42, n_jobs=-1)

# Fit the model
random_search.fit(X_train_scaled, y_train)

# Best parameters found
print(f"Best parameters found: {random_search.best_params_}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
65 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
33 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.11/dist-packages/sklearn/utils/

Best parameters found: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30}


Step 3: Cross-validation Evaluation

In [ ]:
from sklearn.model_selection import cross_val_score

# Using the best model found through RandomizedSearchCV
best_model = random_search.best_estimator_

# Evaluate using cross-validation (e.g., 5-fold)
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')

# Convert negative MSE to positive RMSE for easy interpretation
cv_rmse = (-cv_scores) ** 0.5
print(f"CV RMSE: {cv_rmse.mean():.2f} +/- {cv_rmse.std():.2f}")


CV RMSE: 0.02 +/- 0.02


Step 4: Final Model Evaluation on Test Data

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MAE
mae_test = mean_absolute_error(y_test, y_test_pred)

# RMSE (Manually calculate the square root of MSE)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = mse_test ** 0.5

# R²
r2_test = r2_score(y_test, y_test_pred)

# Print the results
print(f"Test MAE: {mae_test:.2f}")
print(f"Test RMSE: {rmse_test:.2f}")
print(f"Test R²: {r2_test:.2f}")


Test MAE: 0.01
Test RMSE: 0.01
Test R²: 0.78


# Let's start with model serialization, where we save the trained model to disk

In [ ]:
import joblib

# Save the model
joblib.dump(model, 'happy_spending_model.pkl')

# To load the model later
loaded_model = joblib.load('happy_spending_model.pkl')


In [ ]:
import os
print(os.path.exists('happy_spending_model.pkl'))


True


In [ ]:
import os
os.listdir()


['.config', 'drive', 'happy_spending_model.pkl', 'sample_data']

In [ ]:
from google.colab import files
files.download('happy_spending_model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive

# Step 1: Mount Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pickle

# Your trained model (example)
# from sklearn.ensemble import RandomForestClassifier
# model = RandomForestClassifier().fit(X_train, y_train)

import os

os.makedirs(gdir_folder_path, exist_ok=True)


# Save the model to Google Drive
with open(f'{gdir_folder_path}/{happy_spending_model_pkl}', 'wb') as f:
    pickle.dump(model, f)

print(f"✅ Model saved to: {gdir_folder_path}/{happy_spending_model_pkl}")


✅ Model saved to: /content/drive/MyDrive/happy_spending_ml/happy_spending_model.pkl


In [ ]:
import pandas as pd

# Step 2: Save X and y to CSV files
X.to_csv(f'{gdir_folder_path}/{input_file_csv}', index=False)
y.to_csv(f'{gdir_folder_path}/{output_file_csv}', index=False)

print("Files saved successfully on Google Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files saved successfully on Google Drive.
